# Matching Engine (CV vs Job Offer)
## Imports

In [5]:
import pdfplumber
import os
import re

## load data

In [4]:
# --- Define the path to test PDF file
cv_path = "../data/cv_sample_cleaned/Data_Science/DS_1.pdf" 

# extract all text from a PDF
def extract_text_from_pdf(file_path):
    full_text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                full_text += page.extract_text() + "\n"
        return full_text
    except Exception as e:
        return f"Error reading the PDF: {e}"

# --- CV into a variable
raw_cv_text = extract_text_from_pdf(cv_path)

# --- Display the results 
print("======================================================")
print("📄 EXTRACTED CV PREVIEW (First 500 characters):")
print("======================================================")
print(raw_cv_text[:500] + "...\n") # Display only the beginning to avoid cluttering the screen


📄 EXTRACTED CV PREVIEW (First 500 characters):

...



### fake opportunity

In [7]:
# --- fake Job Description for test
job_description_text = """
We are looking for a Python Backend Developer (M/F).
Required skills:
- Mastery of Python 3 and the FastAPI or Django framework.
- Good knowledge of SQL databases (PostgreSQL, SQLite).
- Experience with Git, Docker, and CI/CD pipelines.
- Notions in Machine Learning (Pandas, Scikit-learn) are a plus.
The candidate must show team spirit and good communication skills.
"""


print("======================================================")
print("🎯 TARGET JOB DESCRIPTION:")
print("======================================================")
print(job_description_text)

🎯 TARGET JOB DESCRIPTION:

We are looking for a Python Backend Developer (M/F).
Required skills:
- Mastery of Python 3 and the FastAPI or Django framework.
- Good knowledge of SQL databases (PostgreSQL, SQLite).
- Experience with Git, Docker, and CI/CD pipelines.
- Notions in Machine Learning (Pandas, Scikit-learn) are a plus.
The candidate must show team spirit and good communication skills.



## TEXT PREPROCESSING

In [9]:

def clean_text(text):
    """
    Cleanining : 
    - extra spaces
    - special characters, 
    - to lowercase.
    """
    if not text:
        return ""
    
    # Remove newlines and extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters (keep only letters, numbers, and spaces)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Convert to lowercase and strip leading/trailing spaces
    return text.lower().strip()

# Clean both the CV and the Job Description
cv_clean = clean_text(raw_cv_text)
job_clean = clean_text(job_description_text)

print("✅ Preprocessing complete!")
print("Preview of cleaned CV text:", cv_clean[:150], "...")
print("Preview of cleaned job desc:", job_clean, "...")

✅ Preprocessing complete!
Preview of cleaned CV text:  ...
Preview of cleaned job desc: we are looking for a python backend developer  m f   required skills    mastery of python 3 and the fastapi or django framework    good knowledge of sql databases  postgresql  sqlite     experience with git  docker  and ci cd pipelines    notions in machine learning  pandas  scikit learn  are a plus  the candidate must show team spirit and good communication skills ...


## Vectorisation (Embeddings)

In [ ]:
from sentence_transformers import SentenceTransformer

print("⏳ Loading the AI model (this may take a few seconds the first time)...")

# Load the Hugging Face model (Fast and efficient for semantic similarity)
model = SentenceTransformer('all-MiniLM-L6-v2')

# cleaned texts into vectors (embeddings)
cv_embedding = model.encode(cv_clean)
job_embedding = model.encode(job_clean)

print("✅ Model loaded and text vectorized successfully!")
print(f"Vector shape for CV: {cv_embedding.shape} (List of 384 numbers)")

/home/mubuntux/Dev/Briefs/Career-Pathfinder-AI-🚩/backend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading the AI model (this may take a few seconds the first time)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7698.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded and text vectorized successfully!
Vector shape for CV: (384,) (List of 384 numbers)


## Calcul du Score de Matching (Cosine Similarity)

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine similarity : We use .reshape(1, -1) because sklearn expects a 2D array (a list of lists)
similarity_matrix = cosine_similarity(cv_embedding.reshape(1, -1), job_embedding.reshape(1, -1))

# Extract the raw score (between 0 and 1) and convert it to a percentage
match_score = similarity_matrix[0][0] * 100

print("======================================================")
print(f"🎯 AI MATCHING SCORE: {match_score:.2f} %")
print("======================================================")

# Brief interpretation for the user
if match_score > 75:
    print("💡 Excellent match! This CV is highly relevant for the job.")
elif match_score > 40:
    print("💡 Average match. The candidate has some background, but lacks specific keywords.")
else:
    print("💡 Poor match. The CV belongs to a completely different field.")

🎯 AI MATCHING SCORE: 11.05 %
💡 Poor match. The CV belongs to a completely different field.


## Gap Analysis - Missed Skills

In [12]:
# 1. We manually define the core skills required for this specific job offer
# In a real-world scenario (V2), we would use an LLM (like Llama3) to extract this list automatically from the job description.
expected_skills = [
    "python", "fastapi", "django", "sql", "postgresql", 
    "sqlite", "git", "docker", "ci", "cd", 
    "machine learning", "pandas", "scikit-learn"
]

missing_skills = []
found_skills = []

# 2. Check each skill against the cleaned CV text
for skill in expected_skills:
    # We add spaces around the skill to avoid partial matches (e.g., finding "git" inside "digital")
    # A simple 'in' check is used here for the MVP
    if skill in cv_clean:
        found_skills.append(skill)
    else:
        missing_skills.append(skill)

# 3. Display the final JSON-like result
print("======================================================")
print("📊 GAP ANALYSIS RESULTS")
print("======================================================")
print(f"✅ Skills found in CV ({len(found_skills)}):")
for skill in found_skills:
    print(f"   - {skill}")

print(f"\n❌ Missing skills ({len(missing_skills)}):")
for skill in missing_skills:
    print(f"   - {skill}")

📊 GAP ANALYSIS RESULTS
✅ Skills found in CV (0):

❌ Missing skills (13):
   - python
   - fastapi
   - django
   - sql
   - postgresql
   - sqlite
   - git
   - docker
   - ci
   - cd
   - machine learning
   - pandas
   - scikit-learn
